
# 07b — Sentinel-5P CDSE window extract

This notebook performs a lightweight, GitHub-safe Sentinel-5P metadata extraction using the
Copernicus Data Space Ecosystem (CDSE) OData API.

It is designed to:
- authenticate with CDSE username/password
- search Sentinel-5P Level-2 scenes over a small bounding box around a target site
- save a compact scene catalog CSV / Parquet / JSON
- avoid heavy raw-data downloads by default


In [ ]:

SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961

PRODUCT_KEY = "NO2"   # NO2, SO2, CO
TIMELINESS = "OFFL"   # OFFL or NRTI
DATE_FROM = "2024-01-01"
DATE_TO = "2024-12-31"
BUFFER_DEG = 0.20
MAX_SCENES = 100

DOWNLOAD_FIRST_PRODUCT = False
OUTPUT_DIR = "outputs/s5p"


In [ ]:

from pathlib import Path
import os, json
from datetime import datetime, timezone
import requests
import pandas as pd

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Output dir:", Path(OUTPUT_DIR).resolve())
print("UTC now:", datetime.now(timezone.utc).isoformat())


In [ ]:

try:
    import yaml
except Exception:
    yaml = None

site_id_env = os.getenv("AQ26_SITE_ID")
site_name_env = os.getenv("AQ26_SITE_NAME")
lat_env = os.getenv("AQ26_LAT")
lon_env = os.getenv("AQ26_LON")

if site_id_env:
    SITE_ID = site_id_env
if site_name_env:
    SITE_NAME = site_name_env
if lat_env:
    LAT = float(lat_env)
if lon_env:
    LON = float(lon_env)

run_yml = Path("Configs/run.yml")
sites_csv = Path("Configs/sites.csv")

if run_yml.exists() and yaml is not None:
    try:
        cfg = yaml.safe_load(run_yml.read_text(encoding="utf-8")) or {}
        aq = cfg.get("aq26", {}) if isinstance(cfg, dict) else {}
        target = aq.get("target", {}) if isinstance(aq, dict) else {}
        SITE_NAME = target.get("name", SITE_NAME)
        LAT = float(target.get("lat", LAT))
        LON = float(target.get("lon", LON))
        SITE_ID = target.get("site_id", SITE_ID)
        DATE_FROM = str(aq.get("date_from", DATE_FROM))
        DATE_TO = str(aq.get("date_to", DATE_TO))
    except Exception as e:
        print("Config warning (run.yml):", e)

if sites_csv.exists():
    try:
        s = pd.read_csv(sites_csv)
        if {"site_id","lat","lon"}.issubset(s.columns):
            m = s[s["site_id"].astype(str).str.upper() == str(SITE_ID).upper()]
            if not m.empty:
                SITE_NAME = m.iloc[0].get("site_name", SITE_NAME)
                LAT = float(m.iloc[0]["lat"])
                LON = float(m.iloc[0]["lon"])
    except Exception as e:
        print("Config warning (sites.csv):", e)

print("Site:", SITE_ID, SITE_NAME, LAT, LON)
print("Dates:", DATE_FROM, "to", DATE_TO)


In [ ]:

CDSE_USERNAME = os.getenv("CDSE_USERNAME") or os.getenv("CDSE_USER")
CDSE_PASSWORD = os.getenv("CDSE_PASSWORD")

assert CDSE_USERNAME, "Missing CDSE_USERNAME/CDSE_USER secret"
assert CDSE_PASSWORD, "Missing CDSE_PASSWORD secret"

TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
ODATA_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

def get_cdse_token(username: str, password: str) -> str:
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    r = requests.post(TOKEN_URL, data=data, timeout=60)
    r.raise_for_status()
    j = r.json()
    tok = j.get("access_token")
    if not tok:
        raise RuntimeError(f"No access_token in token response: keys={list(j.keys())}")
    return tok

token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)
print("CDSE token acquired:", token[:20] + "...")


In [ ]:

PRODUCT_PATTERNS = {
    ("NO2", "OFFL"): "L2__NO2___",
    ("NO2", "NRTI"): "L2__NO2___",
    ("SO2", "OFFL"): "L2__SO2___",
    ("SO2", "NRTI"): "L2__SO2___",
    ("CO",  "OFFL"): "L2__CO____",
    ("CO",  "NRTI"): "L2__CO____",
}
name_pat = PRODUCT_PATTERNS.get((PRODUCT_KEY.upper(), TIMELINESS.upper()))
assert name_pat, f"Unsupported PRODUCT_KEY/TIMELINESS combo: {PRODUCT_KEY}/{TIMELINESS}"

minx = LON - BUFFER_DEG
maxx = LON + BUFFER_DEG
miny = LAT - BUFFER_DEG
maxy = LAT + BUFFER_DEG

wkt = f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"

flt = (
    "Collection/Name eq 'SENTINEL-5P' "
    f"and ContentDate/Start ge {DATE_FROM}T00:00:00.000Z "
    f"and ContentDate/Start le {DATE_TO}T23:59:59.999Z "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') "
    f"and contains(Name,'{name_pat}') "
    f"and contains(Name,'_{TIMELINESS.upper()}_')"
)

params = {
    "$filter": flt,
    "$top": str(MAX_SCENES),
    "$orderby": "ContentDate/Start asc",
    "$count": "true",
}
headers = {"Authorization": f"Bearer {token}"}

r = requests.get(ODATA_URL, params=params, headers=headers, timeout=120)
r.raise_for_status()
catalog_json = r.json()
value = catalog_json.get("value", [])
print("Scenes returned:", len(value))


In [ ]:

rows = []
for p in value:
    attrs = p.get("Attributes", [])
    attrs_map = {}
    for a in attrs:
        nm = a.get("Name")
        val = a.get("Value") if "Value" in a else a.get("Values")
        if nm:
            attrs_map[nm] = val

    rows.append({
        "site_id": SITE_ID,
        "site_name": SITE_NAME,
        "lat": LAT,
        "lon": LON,
        "product_key": PRODUCT_KEY.upper(),
        "timeliness": TIMELINESS.upper(),
        "product_id": p.get("Id"),
        "name": p.get("Name"),
        "s3_path": p.get("S3Path"),
        "start_utc": p.get("ContentDate", {}).get("Start"),
        "end_utc": p.get("ContentDate", {}).get("End"),
        "published_utc": p.get("PublicationDate"),
        "online": p.get("Online"),
        "size_bytes": p.get("ContentLength"),
        "footprint_wkt": p.get("GeoFootprint"),
        "processor_version": attrs_map.get("processorVersion"),
        "product_type_attr": attrs_map.get("productType"),
        "orbit_number": attrs_map.get("orbitNumber"),
        "relative_orbit": attrs_map.get("relativeOrbitNumber"),
    })

df = pd.DataFrame(rows)
print(df.shape)
display(df.head(20))


In [ ]:

from pathlib import Path

csv_path = Path(OUTPUT_DIR) / f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.csv"
pq_path  = Path(OUTPUT_DIR) / f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.parquet"
js_path  = Path(OUTPUT_DIR) / f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.json"
manifest_path = Path(OUTPUT_DIR) / f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_manifest.json"

df.to_csv(csv_path, index=False)
df.to_parquet(pq_path, index=False)
js_path.write_text(json.dumps(value, indent=2), encoding="utf-8")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "site_id": SITE_ID,
    "site_name": SITE_NAME,
    "lat": LAT,
    "lon": LON,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "buffer_deg": BUFFER_DEG,
    "product_key": PRODUCT_KEY.upper(),
    "timeliness": TIMELINESS.upper(),
    "odata_url": ODATA_URL,
    "scene_count": int(len(df)),
    "outputs": {
        "csv": str(csv_path),
        "parquet": str(pq_path),
        "json": str(js_path),
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", csv_path)
print("Saved:", pq_path)
print("Saved:", js_path)
print("Saved:", manifest_path)


In [ ]:

if DOWNLOAD_FIRST_PRODUCT and not df.empty:
    first_id = df.iloc[0]["product_id"]
    meta_url = f"{ODATA_URL}({first_id})"
    r = requests.get(meta_url, headers=headers, timeout=120)
    r.raise_for_status()
    meta = r.json()
    meta_path = Path(OUTPUT_DIR) / f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_first_product_metadata.json"
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print("Saved first-product metadata:", meta_path)
else:
    print("DOWNLOAD_FIRST_PRODUCT is False or no scenes found.")
